In [0]:
%sql
create catalog if not exists sa_catalog;
use catalog sa_catalog;
create schema if not exists hr_schema;
use schema hr_schema;
show schemas in sa_catalog;

databaseName
default
hr_schema
information_schema


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
schema=StructType([
    StructField("customer_id",StringType(),True),
    StructField("customer_name",StringType(),True),
    StructField("city",StringType(),True),
    StructField("loyalty_tier",StringType(),True),
    StructField("points_balance",IntegerType(),True),
    StructField("is_active",BooleanType(),True),
    StructField("last_updated",TimestampType(),True)
])
df1=spark.read.csv("/Volumes/sa_catalog/hr_schema/sa_mgd_vol/cdc01/CDC01_Retail_Loyalty_Customer_CDC_snapshot.csv",header=True,schema=schema)

In [0]:
display(df1.count())

12

In [0]:
display(df1.filter(col("points_balance").isNull()).count())

1

In [0]:
schema=StructType([
    StructField("event_id",StringType(),True),
    StructField("customer_id",StringType(),True),
    StructField("customer_name",StringType(),True),
    StructField("city",StringType(),True),
    StructField("loyalty_tier",StringType(),True),
    StructField("points_balance",IntegerType(),True),
    StructField("is_active",BooleanType(),True),
    StructField("last_updated",TimestampType(),True),
    StructField("operation",StringType(),True),
])
df2=spark.read.csv("/Volumes/sa_catalog/hr_schema/sa_mgd_vol/cdc01/CDC01_Retail_Loyalty_Customer_CDC_changes.csv",header=True,schema=schema)

In [0]:
display(df2.count())
display(df2.groupBy("operation").count())

5

operation,count
I,2
U,2
D,1


task2

In [0]:
df1=df1.fillna({"points_balance":0})
display(df1.columns)
df1.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("sa_catalog.hr_schema.cdc01_cust")
display(df1.filter(col("points_balance").isNull()).count())
display(spark.sql("select count(distinct customer_id),sum(points_balance) from sa_catalog.hr_schema.cdc01_cust"))

_1
customer_id
customer_name
city
loyalty_tier
points_balance
is_active
last_updated


0

count(DISTINCTcustomer_id),sum(points_balance)
12,2560


task 3

In [0]:
df2.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("sa_catalog.hr_schema.cdc01_events")

In [0]:
display(spark.sql("""merge into sa_catalog.hr_schema.cdc01_cust as target 
                  using sa_catalog.hr_schema.cdc01_events as source on target.customer_id=source.customer_id 
                  when matched and source.operation='D' then delete when matched and source.operation='U' then update 
                  set target.customer_id=source.customer_id,
                  target.customer_name=source.customer_name,
                  target.city=source.city,
                  target.loyalty_tier=source.loyalty_tier,
                  target.points_balance=source.points_balance,
                  target.is_active=source.is_active,
                  target.last_updated=source.last_updated
                  when not matched and operation='I' then insert
                  (customer_id,customer_name,city,loyalty_tier,points_balance,is_active,last_updated) values(source.customer_id,source.customer_name,source.city,source.loyalty_tier,source.points_balance,source.is_active,source.last_updated);"""))
                

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
%sql
delete from sa_catalog.hr_schema.cdc01_cust where customer_id is null;

num_affected_rows
2


In [0]:
%sql
select * from sa_catalog.hr_schema.cdc01_cust

customer_id,customer_name,city,loyalty_tier,points_balance,is_active,last_updated
C001,Anita,Mumbai,SILVER,120,true,2026-10-01T09:00:00.000Z
C003,Charu,Delhi,SILVER,80,true,2026-10-01T09:00:00.000Z
C005,Esha,Mumbai,GOLD,450,true,2026-10-01T09:00:00.000Z
C007,Gita,Pune,BRONZE,40,true,2026-10-01T09:00:00.000Z
C008,Hari,Hyderabad,SILVER,210,true,2026-10-01T09:00:00.000Z
C009,Isha,Mumbai,GOLD,600,true,2026-10-01T09:00:00.000Z
C010,Jai,Chennai,SILVER,160,false,2026-10-01T09:00:00.000Z
C011,Kiran,Delhi,BRONZE,30,true,2026-10-01T09:00:00.000Z
C012,Lata,Hyderabad,GOLD,500,true,2026-10-01T09:00:00.000Z
C013,Maya,Mumbai,BRONZE,70,true,2026-10-02T10:05:00.000Z


In [0]:
%sql

SELECT
    customer_id,
    loyalty_tier,
    points_balance,
    is_active,
    last_updated
FROM sa_catalog.hr_schema.cdc01_cust
WHERE customer_id IN (
    'C002', 'C004', 'C006', 'C013', 'C014'
)
ORDER BY customer_id;


customer_id,loyalty_tier,points_balance,is_active,last_updated
C002,PLATINUM,520,true,2026-10-02T10:00:00.000Z
C006,SILVER,190,true,2026-10-02T10:02:00.000Z
C013,BRONZE,70,true,2026-10-02T10:05:00.000Z
C014,SILVER,110,true,2026-10-02T10:08:00.000Z


In [0]:
%sql

SELECT
    COUNT(*) AS records,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(points_balance) AS total_points
FROM sa_catalog.hr_schema.cdc01_cust;


records,unique_customers,total_points
13,13,3080


In [0]:
%sql
--Query final row count, distinct customer IDs, NULL points count, total points, and a tier-level count/SUM breakdown.
select count(*),count(distinct customer_id),sum(case when points_balance is null then 1 else 0 end),sum(points_balance) from sa_catalog.hr_schema.cdc01_cust;

count(*),count(DISTINCTcustomer_id),sum(CASEWHENpoints_balanceISNULLTHEN1ELSE0END),sum(points_balance)
13,13,0,3080


In [0]:
%sql
SELECT customer_id, loyalty_tier, points_balance, is_active, last_updated
from sa_catalog.hr_schema.cdc01_cust
WHERE customer_id IN ('C002','C004','C006','C013','C014') ORDER BY customer_id;


customer_id,loyalty_tier,points_balance,is_active,last_updated
C002,PLATINUM,520,true,2026-10-02T10:00:00.000Z
C006,SILVER,190,true,2026-10-02T10:02:00.000Z
C013,BRONZE,70,true,2026-10-02T10:05:00.000Z
C014,SILVER,110,true,2026-10-02T10:08:00.000Z


In [0]:
%sql
describe history sa_catalog.hr_schema.cdc01_cust;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-10-19T18:28:11.000Z,73787254141250,sounutshell@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3002818432231409),34f32ebb-884b-4920-8f29-9028ae61e8c2,1019-164405-12p6oz3-v2n,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2557, p25FileSize -> 2549, numDeletionVectorsRemoved -> 1, minFileSize -> 2549, numAddedFiles -> 1, maxFileSize -> 2549, p75FileSize -> 2549, p50FileSize -> 2549, numAddedBytes -> 2549)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
8,2026-10-19T18:28:09.000Z,73787254141250,sounutshell@gmail.com,DELETE,"Map(predicate -> [""isnull(customer_id#15215)""])",null,List(3002818432231409),34f32ebb-884b-4920-8f29-9028ae61e8c2,1019-164405-12p6oz3-v2n,7,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1914, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 1350, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 557)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
7,2026-10-19T18:27:09.000Z,73787254141250,sounutshell@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3002818432231409),60cb5af9-3e1b-42b6-af82-93c727d66670,1019-164405-12p6oz3-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 4810, p25FileSize -> 2557, numDeletionVectorsRemoved -> 1, minFileSize -> 2557, numAddedFiles -> 1, maxFileSize -> 2557, p75FileSize -> 2557, p50FileSize -> 2557, numAddedBytes -> 2557)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
6,2026-10-19T18:27:07.000Z,73787254141250,sounutshell@gmail.com,MERGE,"Map(predicate -> [""(customer_id#14384 = customer_id#14392)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(operation#14399 = D)"",""actionType"":""delete""},{""predicate"":""(operation#14399 = U)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""predicate"":""(operation#14399 = I)"",""actionType"":""insert""}])",null,List(3002818432231409),60cb5af9-3e1b-42b6-af82-93c727d66670,1019-164405-12p6oz3-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2236, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3827, materializeSourceTimeMs -> 2, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1341, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2423)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
5,2026-10-19T18:25:23.000Z,73787254141250,sounutshell@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3002818432231409),2925eed9-44af-48d3-abe7-47a6dac896b0,1019-164405-12p6oz3-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 4885, p25FileSize -> 2574, numDeletionVectorsRemoved -> 1, minFileSize -> 2574, numAddedFiles -> 1, maxFileSize -> 2574, p75FileSize -> 2574, p50FileSize -> 2574, numAddedBytes -> 2574)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
4,2026-10-19T18:25:21.000Z,73787254141250,sounutshell@gmail.com,MERGE,"Map(predicate -> [""(customer_id#13544 = customer_id#13552)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(operation#13559 = 

In [0]:
%sql
select * from sa_catalog.hr_schema.cdc01_cust version as of 8;

customer_id,customer_name,city,loyalty_tier,points_balance,is_active,last_updated
C001,Anita,Mumbai,SILVER,120,true,2026-10-01T09:00:00.000Z
C003,Charu,Delhi,SILVER,80,true,2026-10-01T09:00:00.000Z
C005,Esha,Mumbai,GOLD,450,true,2026-10-01T09:00:00.000Z
C007,Gita,Pune,BRONZE,40,true,2026-10-01T09:00:00.000Z
C008,Hari,Hyderabad,SILVER,210,true,2026-10-01T09:00:00.000Z
C009,Isha,Mumbai,GOLD,600,true,2026-10-01T09:00:00.000Z
C010,Jai,Chennai,SILVER,160,false,2026-10-01T09:00:00.000Z
C011,Kiran,Delhi,BRONZE,30,true,2026-10-01T09:00:00.000Z
C012,Lata,Hyderabad,GOLD,500,true,2026-10-01T09:00:00.000Z
C013,Maya,Mumbai,BRONZE,70,true,2026-10-02T10:05:00.000Z


In [0]:
%sql
select * from sa_catalog.hr_schema.cdc01_cust version as of 0;

customer_id,customer_name,city,loyalty_tier,points_balance,is_active,last_updated
C001,Anita,Mumbai,SILVER,120,true,2026-10-01T09:00:00.000Z
C002,Bharat,Pune,GOLD,320,true,2026-10-01T09:00:00.000Z
C003,Charu,Delhi,SILVER,80,true,2026-10-01T09:00:00.000Z
C004,Dev,Chennai,BRONZE,50,true,2026-10-01T09:00:00.000Z
C005,Esha,Mumbai,GOLD,450,true,2026-10-01T09:00:00.000Z
C006,Farhan,Delhi,SILVER,0,true,2026-10-01T09:00:00.000Z
C007,Gita,Pune,BRONZE,40,true,2026-10-01T09:00:00.000Z
C008,Hari,Hyderabad,SILVER,210,true,2026-10-01T09:00:00.000Z
C009,Isha,Mumbai,GOLD,600,true,2026-10-01T09:00:00.000Z
C010,Jai,Chennai,SILVER,160,false,2026-10-01T09:00:00.000Z


In [0]:
%sql
describe detail sa_catalog.hr_schema.cdc01_cust;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1c3cc785-b3cd-4c9e-8eb9-7be7e54de8bf,sa_catalog.hr_schema.cdc01_cust,null,,2026-10-19T17:51:10.317Z,2026-10-19T18:28:11.000Z,List(),List(),1,2549,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 0b0ef57f-07f2-4cee-9995-5916bdedb833)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
describe table extended sa_catalog.hr_schema.cdc01_cust

col_name,data_type,comment
customer_id,string,null
customer_name,string,null
city,string,null
loyalty_tier,string,null
points_balance,int,null
is_active,boolean,null
last_updated,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"city, is_active, last_updated, points_balance, loyalty_tier, customer_id, customer_name",


In [0]:
%sql
SELECT loyalty_tier, COUNT(*) AS customers, SUM(points_balance) AS total_points
FROM sa_catalog.hr_schema.cdc01_cust
GROUP BY loyalty_tier ORDER BY loyalty_tier;


loyalty_tier,customers,total_points
BRONZE,3,140
GOLD,3,1550
PLATINUM,1,520
SILVER,6,870
